# hls4ml → Felix-155 (xcvp1552)

Train a jet-classifier MLP, convert with hls4ml, compile to the custom platform, produce an SD card image.

| Step | What |
|------|------|
| §1 Train | Keras MLP 16→64→32→32→5, jet tagging (5 classes) |
| §2 hls4ml | Convert to ap_fixed<16,6>, C-sim accuracy check |
| §3 Build | `v++` compile + link via custom platform |
| §4 Package | Cross-compile aarch64 host, assemble `sd_card/` |

Kernel sources live in `src/` — edit them there, not here.

```bash
# Before running:
source /tools/Xilinx/Vitis/2024.2/settings64.sh && source /opt/xilinx/xrt/setup.sh
cd step6_vp1552/ && jupyter lab
```

In [1]:
import shutil, os, pathlib, subprocess, sys, time
import numpy as np

assert shutil.which('v++'), 'v++ not found — source Vitis settings64.sh'

_here    = pathlib.Path.cwd()   # must be step6_vp1552/
PLATFORM = os.environ.get(
    'VERSAL_XPFM',
    str(_here.parent / 'step3_vp1552/ws/custom_platform/export/custom_platform/custom_platform.xpfm'),
)
assert os.path.exists(PLATFORM), f'Platform not found: {PLATFORM}'

TARGET  = 'hw'    # 'hw_emu' for emulation
HLS_DIR = 'hls4ml_prj'
SRC     = _here / 'src'

print(f'Platform : {PLATFORM}')
print(f'Target   : {TARGET}')

Platform : /home/synthara/VersalPrjs/felix/felix-xpfm-project/step3_vp1552/ws/custom_platform/export/custom_platform/custom_platform.xpfm
Target   : hw


---
## §1  Train

In [2]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score
from tensorflow.keras.utils import to_categorical
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l1
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

data  = fetch_openml('hls4ml_lhc_jets_hlf', as_frame=False, cache=True)
X, y  = data['data'], data['target']
le    = LabelEncoder()
y     = to_categorical(le.fit_transform(y), 5)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
sc    = StandardScaler()
X_tr  = sc.fit_transform(X_tr)
X_te  = sc.transform(X_te)

MODEL_DIR = 'nn_model'
if not os.path.exists(f'{MODEL_DIR}/model.h5'):
    tf.random.set_seed(0)
    model = Sequential([
        Dense(64, input_shape=(16,), name='fc1', kernel_regularizer=l1(1e-4)), Activation('relu'),
        Dense(32, name='fc2',        kernel_regularizer=l1(1e-4)), Activation('relu'),
        Dense(32, name='fc3',        kernel_regularizer=l1(1e-4)), Activation('relu'),
        Dense(5,  name='output',     kernel_regularizer=l1(1e-4)), Activation('softmax'),
    ])
    model.compile(optimizer=Adam(1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
    model.fit(X_tr, y_tr, batch_size=1024, epochs=30, validation_split=0.2,
              callbacks=[EarlyStopping(patience=8, restore_best_weights=True),
                         ReduceLROnPlateau(factor=0.5, patience=4, min_lr=1e-6)])
    os.makedirs(MODEL_DIR, exist_ok=True)
    model.save(f'{MODEL_DIR}/model.h5')
else:
    model = tf.keras.models.load_model(f'{MODEL_DIR}/model.h5')
    print(f'Loaded {MODEL_DIR}/model.h5')

y_pred = model.predict(X_te, verbose=0)
print(f'Keras accuracy : {accuracy_score(np.argmax(y_te,1), np.argmax(y_pred,1)):.4f}')

2026-02-27 14:00:00.359044: I tensorflow/core/util/port.cc:111] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-27 14:00:00.383879: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/synthara/.conda/envs/hls4ml-tutorial/lib/python3.10/site-packages/sklearn/datasets/_openml.py:968: FutureWarning: The default value of `parser` will change from `'liac-arff'` to `'auto'` in 1.4. You can set `parser='auto'` to silence this warning. Therefore, an `ImportError` will be raised from 1.4 if the dataset is dense and pandas is 

Loaded nn_model/model.h5
Keras accuracy : 0.7628


---
## §2  hls4ml convert + C-sim check

`hls_model.build()` is skipped — xcvp1552 is not in the standalone Vitis HLS device DB.  
Real HLS synthesis runs inside `v++` in §3 via the platform XSA.

In [3]:
import hls4ml

cfg = hls4ml.utils.config_from_keras_model(model, granularity='model', backend='Vitis')
cfg['Model']['Precision']   = 'ap_fixed<16,6>'
cfg['Model']['ReuseFactor'] = 1

hls_model = hls4ml.converters.convert_from_keras_model(
    model,
    hls_config = cfg,
    backend    = 'Vitis',
    output_dir = HLS_DIR,
    part       = 'xcvp1552-vsva3340-2MHP-e-S',
    io_type    = 'io_parallel',
)
print(f'Firmware → {HLS_DIR}/firmware/')

# C-sim: verify fixed-point accuracy before the long v++ run
hls_model.compile()
y_hls = hls_model.predict(np.ascontiguousarray(X_te[:1000], dtype=np.float32))
acc_k = accuracy_score(np.argmax(y_te[:1000],1), np.argmax(y_pred[:1000],1))
acc_h = accuracy_score(np.argmax(y_te[:1000],1), np.argmax(y_hls,       1))
print(f'Keras  : {acc_k:.4f}')
print(f'hls4ml : {acc_h:.4f}  (ap_fixed<16,6>)')
assert abs(acc_k - acc_h) < 0.02, 'Accuracy drop > 2% — widen precision or raise ReuseFactor'

Firmware → hls4ml_prj/firmware/
Keras  : 0.7920
hls4ml : 0.7860  (ap_fixed<16,6>)


---
## §3  Build  (`v++` compile + link)

Reads `src/nn_top.cpp` — the static Vitis DDR wrapper for `myproject()`.  
**hw build: ~15–25 min.** Switch `TARGET = 'hw_emu'` above for a fast smoke-test.

In [4]:
sys.path.insert(0, str(_here))
from vitis_build import VitisKernel

vk = VitisKernel(platform=PLATFORM)

t0 = time.time()
nn_xclbin = vk.build_hls4ml(
    hls4ml_dir  = HLS_DIR,
    wrapper_src = (SRC / 'nn_top.cpp').read_text(),
    kernel_name = 'nn_top',
    target      = TARGET,
    clean       = False,   # True to wipe a stale failed build
)
print(f'Done in {(time.time()-t0)/60:.1f} min  →  {nn_xclbin}')

Platform     : /home/synthara/VersalPrjs/felix/felix-xpfm-project/step3_vp1552/ws/custom_platform/export/custom_platform/custom_platform.xpfm
v++          : /tools/Xilinx/Vitis/2024.2/bin/v++
Build dir    : /home/synthara/VersalPrjs/felix/felix-xpfm-project/step6_vp1552/vitis_builds
rootfs       : /home/synthara/VersalPrjs/felix/felix-xpfm-project/step2_vp1552/my_foe_flx/images/linux/rootfs.ext4
kernel_image : /home/synthara/VersalPrjs/felix/felix-xpfm-project/step2_vp1552/my_foe_flx/images/linux/Image
[1/3] v++ -c  (hw) nn_top.cpp -> .xo ...
       Compile done (164s)
[2/3] v++ -l  (hw) .xo -> .xsa ...
  >> Check VPL, containing 6 checks, has run: 0 errors
  >> WARNING: Skipping CLOCK_FREQ_TOPOLOGY section for count size is zero.
  >> WARNING: Section 'CLOCK_FREQ_TOPOLOGY' content is empty.  No data in the given JSON file.
  >> Check POST-VPL, containing 1 checks, has run: 0 errors
  >> WARNING: [v++ 60-1628] No sd_card image will be generated for Versal platforms. Parameter compiler.

---
## §4  Package SD card

Cross-compiles `src/nn_host.cpp` for aarch64, then re-runs `v++ -p` to bake  
the host binary into the SD card image alongside the xclbin.

In [6]:
SYSROOT   = str(_here.parent / 'step2_vp1552/my_foe_flx/images/linux/sdk/sysroots/cortexa72-cortexa53-xilinx-linux')
CROSS_CXX = os.path.join(os.environ['XILINX_VITIS'], 'gnu/aarch64/lin/aarch64-linux/bin/aarch64-linux-gnu-g++')
HOST_BIN  = str(_here / 'nn_host_aarch64')

# Cross-compile host
r = subprocess.run([
    CROSS_CXX, '-O2', '-std=c++14',
    f'-I{SYSROOT}/usr/include/xrt',
    f'-I{os.environ["XILINX_VIVADO"]}/include',
    f'--sysroot={SYSROOT}',
    '-o', HOST_BIN, str(SRC / 'nn_host.cpp'),
    f'-L{SYSROOT}/usr/lib', '-lxilinxopencl', '-pthread', '-lrt',
], capture_output=True, text=True)
if r.returncode: print(r.stderr); raise RuntimeError('host compile failed')
print(f'Host : {HOST_BIN}')

# Package: bake host binary into sd_card
nn_xsa = str(pathlib.Path(nn_xclbin).with_suffix('.xsa'))
nn_xclbin = vk.package(
    xsa_path       = nn_xsa,
    kernel_name    = 'nn_top',
    target         = TARGET,
    extra_sd_files = [HOST_BIN],
)

# Show result
sd = pathlib.Path(nn_xclbin).parent / 'package' / 'sd_card'
if sd.is_dir():
    print(f'\nsd_card/:')
    for f in sorted(sd.iterdir()):
        print(f'  {f.name:<28} {f.stat().st_size//1024:>6} KB')

imgs = list(pathlib.Path(nn_xclbin).parent.rglob('sd_card.img'))
print(f'\nFlash: sudo dd if={imgs[0]} of=/dev/sdX bs=4M status=progress && sync' if imgs
      else f'\nNo .img — copy sd_card/* to FAT partition manually.')

Host : /home/synthara/VersalPrjs/felix/felix-xpfm-project/step6_vp1552/nn_host_aarch64
[3/3] v++ -p  (hw) .xsa -> .xclbin ...
  + sd_file: /home/synthara/VersalPrjs/felix/felix-xpfm-project/step6_vp1552/nn_host_aarch64
  >> WARNING: [v++ 82-10536] Platform doesn't contain boot mode
Build complete in 0.4 min
  xclbin  : /home/synthara/VersalPrjs/felix/felix-xpfm-project/step6_vp1552/vitis_builds/nn_top_hw/nn_top.xclbin (14470 KB)
  log     : /home/synthara/VersalPrjs/felix/felix-xpfm-project/step6_vp1552/vitis_builds/nn_top_hw/build.log
  sd_card : /home/synthara/VersalPrjs/felix/felix-xpfm-project/step6_vp1552/vitis_builds/nn_top_hw/package/sd_card/
            BOOT.BIN
            Image
            boot.scr
            nn_host_aarch64
            nn_top.xclbin

sd_card/:
  BOOT.BIN                       7842 KB
  Image                         24040 KB
  boot.scr                          3 KB
  nn_host_aarch64                  80 KB
  nn_top.xclbin                 14469 KB

Flash: sudo

---
## §5  Boot & run

```bash
# Write SD card
sudo dd if=<path>/sd_card.img of=/dev/sdX bs=4M status=progress && sync
# — or copy sd_card/* to FAT partition if no .img

# SSH into board after boot
ssh root@192.168.1.100

/boot/nn_host /boot/nn_top.xclbin        # 8 embedded test samples
/boot/nn_host /boot/nn_top.xclbin 64     # cycle through samples 64 times
```

Expected:
```
Device : xilinx_vp1552_...
Sample  Pred  GT    Scores
--------------------------------------------------
0       q     q     0.031 0.912 0.021 0.018 0.018 [OK]
...
Accuracy : 7/8 = 87%    Kernel : 0.52 ms (0.065 ms/sample)
```

Hot-swap xclbin without re-burning:
```bash
scp vitis_builds/nn_top_hw/nn_top.xclbin root@192.168.1.100:/tmp/
/boot/nn_host /tmp/nn_top.xclbin
```